<a href="https://colab.research.google.com/github/thanhf16hehe-droid/tuan2_phanchithanh/blob/main/tuan3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Cài thư viện và chuẩn bị dữ liệu

In [12]:
%pip -q install imbalanced-learn

import numpy as np
import pandas as pd

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import (
    train_test_split, KFold, StratifiedKFold, cross_validate
)
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    make_scorer, precision_score, recall_score, f1_score,
    average_precision_score
)

from imblearn.pipeline import Pipeline
from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.under_sampling import RandomUnderSampler

SEED = 42

cancer = load_breast_cancer(as_frame=True)
X_all = cancer.data.copy()

# scikit-learn gốc: 0 = malignant, 1 = benign
# Đổi nhãn theo notebook mẫu: 1 = malignant, 0 = benign
y_all = (cancer.target == 0).astype(int)

benign_idx = y_all[y_all == 0].index
malignant_idx = y_all[y_all == 1].index

print("Benign:", len(benign_idx))
print("Malignant trong dữ liệu gốc:", len(malignant_idx))

def make_dataset(n_malignant=60):
    """Giữ toàn bộ benign, lấy n_malignant malignant.
       None nghĩa là dùng toàn bộ malignant gốc."""
    if n_malignant is None:
        selected_malignant = malignant_idx.to_numpy()
    else:
        selected_malignant = malignant_idx.to_series().sample(
            n=n_malignant, random_state=SEED
        ).to_numpy()

    selected = np.concatenate([benign_idx.to_numpy(), selected_malignant])
    selected = pd.Series(selected).sample(
        frac=1, random_state=SEED
    ).to_numpy()

    return (
        X_all.loc[selected].reset_index(drop=True),
        y_all.loc[selected].reset_index(drop=True)
    )

def split_data(n_malignant=60):
    X, y = make_dataset(n_malignant)
    return train_test_split(
        X, y, test_size=0.20, random_state=SEED, stratify=y
    )

scoring = {
    "accuracy": "accuracy",
    "precision": make_scorer(precision_score, zero_division=0),
    "recall": make_scorer(recall_score, zero_division=0),
    "f1": make_scorer(f1_score, zero_division=0),
    "pr_auc": "average_precision"
}

def make_model(classifier=None, sampler=None, scale=True):
    steps = [("scaler", StandardScaler() if scale else "passthrough")]
    if sampler is not None:
        steps.append(("sampler", sampler))
    if classifier is None:
        classifier = LogisticRegression(max_iter=5000, random_state=SEED)
    steps.append(("model", classifier))
    return Pipeline(steps)

def cv_scores(model, X_train, y_train, cv):
    result = cross_validate(
        model, X_train, y_train, cv=cv, scoring=scoring
    )
    return {
        metric: (
            result[f"test_{metric}"].mean(),
            result[f"test_{metric}"].std()
        )
        for metric in scoring
    }

Benign: 357
Malignant trong dữ liệu gốc: 212


Bài 1 — Thay đổi mức độ imbalance

In [13]:
from sklearn.metrics import accuracy_score

rows = []

for n_malignant in [40, 120, None]:
    X_train, X_test, y_train, y_test = split_data(n_malignant)

    model = make_model()
    model.fit(X_train, y_train)

    pred = model.predict(X_test)
    prob = model.predict_proba(X_test)[:, 1]

    rows.append({
        "Malignant giữ lại": "Toàn bộ" if n_malignant is None else n_malignant,
        "Tỷ lệ malignant (%)": round(100 * y_test.mean(), 2),
        "Accuracy": accuracy_score(y_test, pred),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1": f1_score(y_test, pred, zero_division=0),
        "PR-AUC": average_precision_score(y_test, prob)
    })

pd.DataFrame(rows).set_index("Malignant giữ lại").round(4)

,Tỷ lệ malignant (%),Accuracy,Recall,F1,PR-AUC
Malignant giữ lại,,,,,
40,10.00,1.0000,1.0000,1.0000,1.0000
120,25.00,0.9688,0.9167,0.9362,0.9567
Toàn bộ,36.84,0.9649,0.9286,0.9512,0.9888


So sánh Recall và PR-AUC cùng Accuracy. Khi lớp malignant ít đi, Accuracy vẫn có thể trông tốt dù mô hình bỏ sót nhiều malignant

Bài 2 — KFold và StratifiedKFold

In [14]:
X_train, X_test, y_train, y_test = split_data(60)

cv_options = {
    "KFold": KFold(n_splits=5, shuffle=True, random_state=SEED),
    "StratifiedKFold": StratifiedKFold(
        n_splits=5, shuffle=True, random_state=SEED
    )
}

rows = []
for cv_name, cv in cv_options.items():
    result = cv_scores(make_model(), X_train, y_train, cv)
    for metric, (mean, std) in result.items():
        rows.append({
            "CV": cv_name, "Metric": metric,
            "Mean": mean, "Std": std
        })

pd.DataFrame(rows).pivot(
    index="Metric", columns="CV", values=["Mean", "Std"]
).round(4)

Mean                     Std                
CV          KFold StratifiedKFold   KFold StratifiedKFold
Metric                                                   
accuracy   0.9880          0.9850  0.0112          0.0164
f1         0.9446          0.9447  0.0613          0.0646
pr_auc     0.9891          0.9855  0.0188          0.0291
precision  1.0000          0.9618  0.0000          0.0469
recall     0.9012          0.9400  0.1052          0.1200

StratifiedKFold cố giữ tỷ lệ hai lớp gần nhau giữa các fold. Khi malignant ít, điều này giúp mỗi fold có số mẫu malignant tương đối đồng đều và kết quả thường ổn định hơn

Bài 3 — Thay đổi số fold

In [15]:
rows = []

for k in [3, 5, 10]:
    cv = StratifiedKFold(
        n_splits=k, shuffle=True, random_state=SEED
    )
    result = cv_scores(make_model(), X_train, y_train, cv)

    for metric, (mean, std) in result.items():
        rows.append({
            "K": k, "Metric": metric,
            "Mean": mean, "Std": std
        })

pd.DataFrame(rows).pivot(
    index="Metric", columns="K", values=["Mean", "Std"]
).round(4)

Mean                     Std                
K              3       5       10      3       5       10
Metric                                                   
accuracy   0.9880  0.9850  0.9880  0.0042  0.0164  0.0147
f1         0.9563  0.9447  0.9564  0.0162  0.0646  0.0550
pr_auc     0.9907  0.9855  0.9861  0.0132  0.0291  0.0417
precision  1.0000  0.9618  0.9667  0.0000  0.0469  0.0667
recall     0.9167  0.9400  0.9550  0.0295  0.1200  0.0907

Nhận xét sự thay đổi của mean và std. K lớn hơn không đảm bảo điểm số tốt hơn; mỗi fold validation nhỏ hơn, nên metric có thể dao động nhiều hơn

Bài 4 — Thay đổi mức Over-Sampling

In [16]:
cv5 = StratifiedKFold(
    n_splits=5, shuffle=True, random_state=SEED
)

rows = []
for ratio in [0.5, 0.75, 1.0]:
    model = make_model(
        sampler=RandomOverSampler(
            sampling_strategy=ratio, random_state=SEED
        )
    )
    result = cv_scores(model, X_train, y_train, cv5)

    row = {"sampling_strategy": ratio}
    for metric, (mean, std) in result.items():
        row[metric + "_mean"] = mean
        row[metric + "_std"] = std
    rows.append(row)

pd.DataFrame(rows).set_index("sampling_strategy").round(4)

,accuracy_mean,accuracy_std,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std,pr_auc_mean,pr_auc_std
sampling_strategy,,,,,,,,,,
0.50,0.988,0.0112,0.9618,0.0469,0.96,0.08,0.9577,0.0411,0.9836,0.0284
0.75,0.988,0.0112,0.9618,0.0469,0.96,0.08,0.9577,0.0411,0.9855,0.0291
1.00,0.985,0.0094,0.9418,0.0476,0.96,0.08,0.9472,0.0353,0.9855,0.0291


sampling_strategy=1.0 cân bằng số mẫu hai lớp sau khi lấy mẫu. Oversampling mạnh hơn có thể tăng Recall nhưng cũng có thể làm giảm Precision; hãy kiểm tra kết quả thực tế trong bảng.

Bài 5 — So sánh ba kỹ thuật sampling

In [ ]:
samplers = {
    "Random Over-Sampling": RandomOverSampler(random_state=SEED),
    "Random Under-Sampling": RandomUnderSampler(random_state=SEED),
    "SMOTE": SMOTE(random_state=SEED)
}

rows = []
for name, sampler in samplers.items():
    result = cv_scores(make_model(sampler=sampler), X_train, y_train, cv5)
    row = {"Phương pháp": name}
    for metric, (mean, std) in result.items():
        row[metric + "_mean"] = mean
        row[metric + "_std"] = std
    rows.append(row)

pd.DataFrame(rows).set_index("Phương pháp").round(4)

So sánh Precision, Recall, F1 và PR-AUC, không chỉ Accuracy. Under-Sampling bỏ bớt mẫu benign; Over-Sampling lặp lại mẫu malignant; SMOTE tạo mẫu malignant tổng hợp

Bài 6 — Thay classifier và kiểm tra StandardScaler
Giữ SMOTE, thay Logistic Regression bằng KNN, SVM và Decision Tree. Mỗi classifier được so sánh có và không có scaler trên cùng folds

In [ ]:
classifiers = {
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "SVM": SVC(kernel="rbf", probability=True, random_state=SEED),
    "Decision Tree": DecisionTreeClassifier(random_state=SEED)
}

rows = []
for name, classifier in classifiers.items():
    for use_scaler in [True, False]:
        model = make_model(
            classifier=classifier,
            sampler=SMOTE(random_state=SEED),
            scale=use_scaler
        )
        result = cv_scores(model, X_train, y_train, cv5)

        row = {
            "Classifier": name,
            "StandardScaler": "Có" if use_scaler else "Không"
        }
        for metric, (mean, std) in result.items():
            row[metric + "_mean"] = mean
            row[metric + "_std"] = std
        rows.append(row)

pd.DataFrame(rows).set_index(
    ["Classifier", "StandardScaler"]
).round(4)

KNN và SVM thường nhạy với scale vì dựa vào khoảng cách hoặc hình học của dữ liệu. Decision Tree thường ít nhạy hơn với việc chuẩn hóa. Hãy dùng bảng để xem mức khác biệt trong lần chạy này

Bài 7 — Đáp án tham khảo
1. Không sampling test vì test cần mô phỏng dữ liệu thật chưa thấy; sampling làm thay đổi phân bố lớp.
2. Không fit scaler trên toàn bộ dữ liệu vì scaler sẽ dùng thông tin từ validation/test, gây data leakage.
3. Recall quan trọng hơn Precision khi bỏ sót mẫu malignant gây hậu quả lớn.
4. Precision quan trọng hơn Recall khi dự đoán nhầm malignant gây tốn kém hoặc dẫn đến xử lý không cần thiết.
5. PR-AUC hữu ích với dữ liệu mất cân bằng vì tập trung vào Precision và Recall của lớp dương.
6. StratifiedKFold phù hợp hơn vì cố giữ tỷ lệ lớp gần nhau trong từng fold.
7. Under-Sampling có thể mất thông tin vì loại bỏ ngẫu nhiên mẫu benign, có thể gồm các mẫu quan trọng.
8. SMOTE khác Random Over-Sampling: Random Over-Sampling sao chép mẫu malignant; SMOTE tạo mẫu tổng hợp dựa trên các mẫu malignant lân cận.